<a href="https://colab.research.google.com/github/abdelruhman161-cyber/Assignment-1/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelruhman161-cyber/Assignment-1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import json
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

# 1. Connection setup
try:
    hf_token_val = userdata.get('HF_TOKEN')
except Exception:
    hf_token_val = os.environ.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{hf_token_val}');")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_PERFORMANCE = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Detect correct date column dynamically
schema_df = con.execute(f"DESCRIBE SELECT * FROM {DIM_CONTENT}").df()
cols = [c.lower() for c in schema_df['column_name'].tolist()]
date_col = 'first_seen_date' if 'first_seen_date' in cols else ('published_at' if 'published_at' in cols else 'content_created_date')

# 2. Extract feature dataset using March 2026 data and April 2026 ground truth label
query_data = f"""
WITH m3 AS (
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        COALESCE(DATEDIFF('day', TRY_CAST(c.{date_col} AS DATE), DATE '2026-03-31'), 0) as page_age_days,
        SUM(f.gsc_impressions) as imp_m3,
        SUM(f.gsc_clicks) as clicks_m3,
        AVG(f.gsc_sum_position) as pos_m3,
        CASE WHEN SUM(f.gsc_impressions) > 0 THEN (SUM(f.gsc_clicks)::FLOAT / SUM(f.gsc_impressions)) ELSE 0 END as ctr_m3
    FROM {FACT_PERFORMANCE} f
    LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE STRFTIME(f.report_date, '%Y-%m') = '2026-03'
    GROUP BY f.content_hash_id, f.client_hash_id, c.{date_col}
),
m4 AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as imp_m4
    FROM {FACT_PERFORMANCE}
    WHERE STRFTIME(report_date, '%Y-%m') = '2026-04'
    GROUP BY content_hash_id
)
SELECT
    m3.*,
    CASE
        WHEN m4.imp_m4 IS NULL OR m3.imp_m3 = 0 THEN 0
        WHEN ((m4.imp_m4 - m3.imp_m3)::FLOAT / m3.imp_m3) < -0.20 THEN 1
        ELSE 0
    END as is_declining
FROM m3
LEFT JOIN m4 ON m3.content_hash_id = m4.content_hash_id
WHERE m3.imp_m3 >= 100;
"""

df_model = con.execute(query_data).df().fillna(0)
print(f"✅ Extracted dataset: {df_model.shape[0]:,} rows across {df_model['client_hash_id'].nunique()} unique clients.")

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# Calculate W04 Baseline Score on feature set
df_model['baseline_score'] = (
    np.log1p(df_model['imp_m3']) * 0.4 +
    (df_model['page_age_days'] / 365.0).clip(0, 3) * 0.4 -
    df_model['ctr_m3'] * 0.2
)

# Define Features and Target
feature_cols = ['imp_m3', 'clicks_m3', 'pos_m3', 'ctr_m3', 'page_age_days']
X = df_model[feature_cols]
y = df_model['is_declining']
groups = df_model['client_hash_id']

# GroupShuffleSplit to avoid client domain leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
baseline_val = df_model.iloc[val_idx]['baseline_score']

print(f"Train set: {X_train.shape[0]:,} rows ({groups.iloc[train_idx].nunique()} clients)")
print(f"Validation set: {X_val.shape[0]:,} rows ({groups.iloc[val_idx].nunique()} clients)")

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# 1. Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 2. Predict Probabilities on Validation Set
y_pred_rf = rf_model.predict_proba(X_val)[:, 1]

# 3. Compute Metrics
rf_roc_auc = roc_auc_score(y_val, y_pred_rf)
rf_pr_auc = average_precision_score(y_val, y_pred_rf)

base_roc_auc = roc_auc_score(y_val, baseline_val)
base_pr_auc = average_precision_score(y_val, baseline_val)

# 4. Print Comparison Table
comparison_df = pd.DataFrame({
    'Model / Method': ['Week 4 Baseline Rule', 'Week 5 Random Forest'],
    'ROC-AUC': [base_roc_auc, rf_roc_auc],
    'PR-AUC': [base_pr_auc, rf_pr_auc]
})

print("=== MODEL VS BASELINE PERFORMANCE COMPARISON ===")
print(comparison_df.to_string(index=False))

# Export Metrics JSON receipt
os.makedirs('../outputs', exist_ok=True)
metrics_payload = {
    "baseline_roc_auc": float(base_roc_auc),
    "model_roc_auc": float(rf_roc_auc),
    "baseline_pr_auc": float(base_pr_auc),
    "model_pr_auc": float(rf_pr_auc),
    "val_sample_size": int(len(y_val))
}
with open('../outputs/w05_model_metrics.json', 'w') as f:
    json.dump(metrics_payload, f, indent=4)

print("\n✅ Metrics saved to work/outputs/w05_model_metrics.json")

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# 1. Feature Importance Analysis
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("--- Feature Importances ---")
print(feature_importance.to_string(index=False))

# 2. Error Analysis (False Positives & False Negatives)
val_results = X_val.copy()
val_results['actual'] = y_val
val_results['pred_prob'] = y_pred_rf
val_results['error'] = np.abs(val_results['actual'] - val_results['pred_prob'])

top_errors = val_results.sort_values(by='error', ascending=False).head(5)
print("\n--- Top 5 Inspection Errors ---")
print(top_errors[['imp_m3', 'page_age_days', 'ctr_m3', 'actual', 'pred_prob']])


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.